In [2]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import mysql.connector

app = Flask(__name__)
CORS(app)

def get_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        user="root",
        password="1234",
        database="timeWise"
    )

@app.route('/login', methods=['POST'])
def login():
    data = request.get_json()

    username = data.get('username')
    password = data.get('password')

    conn = get_connection()
    cursor = conn.cursor()

    query = "SELECT * FROM Akun WHERE username=%s AND password=%s"
    cursor.execute(query, (username, password))

    result = cursor.fetchone()
    conn.close()

    return jsonify({"status": "success" if result else "fail"})


@app.route('/register', methods=['POST'])
def register():
    data = request.get_json()

    username = data.get('username')
    password = data.get('password')

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM Akun WHERE username=%s", (username,))
    if cursor.fetchone():
        return jsonify({"status": "username_taken"})

    cursor.execute(
        "INSERT INTO Akun (idAkun, username, password) VALUES (NULL, %s, %s)",
        (username, password)
    )
    conn.commit()
    conn.close()

    return jsonify({"status": "registered"})


if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import mysql.connector

class Koneksi:
    @staticmethod
    def get_connection():
        return mysql.connector.connect(
            host="127.0.0.1",
            user="root",
            password="1234",
            database="timeWise"
        )
        
class Akun:
    def __init__(self, idAkun, username, password):
        self.idAkun = idAkun
        self.username = username
        self.password = password

class Jadwal:
    def __init__(self, idJadwal, namaJadwal, tanggal, waktu, prioritas, deadline):
        self.idJadwal = idJadwal
        self.namaJadwal = namaJadwal
        self.tanggal = tanggal
        self.waktu = waktu
        self.prioritas = prioritas
        self.deadline = deadline

class Laporan:
    def __init__(self, idLaporan, jenis):
        self.idLaporan = idLaporan
        self.jenis = jenis

class Notifikasi:
    def __init__(self, idNotifikasi, jenis):
        self.idNotifikasi = idNotifikasi
        self.jenis = jenis

class LoginService:

    def login(self, username, password):
        conn = Koneksi.get_connection()
        cursor = conn.cursor()

        query = "SELECT * FROM Akun WHERE username=%s AND password=%s"
        cursor.execute(query, (username, password))

        result = cursor.fetchone()
        conn.close()

        return result is not None

    def register(self, username, password):
        conn = Koneksi.get_connection()
        cursor = conn.cursor()

        # cek username sudah ada atau belum
        check_query = "SELECT * FROM Akun WHERE username=%s"
        cursor.execute(check_query, (username,))
        
        if cursor.fetchone():
            print("Username sudah digunakan!")
            conn.close()
            return False

        # ambil id baru (auto increment manual)
        cursor.execute("SELECT MAX(idAkun) FROM Akun")
        last_id = cursor.fetchone()[0]
        new_id = 1 if last_id is None else last_id + 1

        # insert akun baru
        insert_query = "INSERT INTO Akun (idAkun, username, password) VALUES (%s, %s, %s)"
        cursor.execute(insert_query, (new_id, username, password))
        conn.commit()

        conn.close()
        print("Akun berhasil dibuat!")
        return True

In [ ]:
if __name__ == "__main__":
    login_service = LoginService()

    while True:
        print("\n=== MENU ===")
        print("1. Sudah punya akun (Login)")
        print("2. Belum punya akun (Register)")
        print("3. Keluar")

        pilihan = input("Pilih menu (1/2/3): ")

        if pilihan == "1":
            username = input("Username: ")
            password = input("Password: ")

            if login_service.login(username, password):
                print("Login berhasil!")
                break
            else:
                print("Login gagal!")

        elif pilihan == "2":
            username = input("Buat Username: ")
            password = input("Buat Password: ")

            login_service.register(username, password)

        elif pilihan == "3":
            print("Program selesai.")
            break

        else:
            print("Pilihan tidak valid!")